# Laboratorul 4 — DDPM de la zero

**Modele generative • Fashion-MNIST • Notebook de lucru și raportare**

Notebook-ul urmează cerințele din „Laboratoare Modele de difuzie”, Laboratorul 4. Rulați celulele în ordine. Rezultatele numerice și vizuale sunt produse numai prin rularea experimentelor; nu sunt incluse măsurători fabricate. Completați evaluările vizuale și răspunsurile de control înainte de exportul raportului PDF.

**Obiective:** proces direct, U-Net condiționat de timp, antrenare cu programe liniar/cosinus, eșantionare DDPM și DDIM, comparație și raport de 2–4 pagini.

## Înainte de rulare — Colab, GPU și timp

1. Deschideți notebook-ul în Google Colab și alegeți **Runtime → Change runtime type → T4 GPU**. Este acceptat și un GPU CUDA local. Celula de configurare verifică dispozitivul; CPU este potrivit pentru verificări, dar antrenarea și eșantionarea la 1000 de pași pot dura foarte mult.
2. Runtime-ul poate fi deconectat/resetat. Salvați notebook-ul în Drive. Pentru output persistent în Colab, rulați înainte de inițializarea rulării 

In [ ]:
import os, sys, json, math, time, uuid, logging, platform, random
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED, T, BATCH_SIZE, EPOCHS = 42, 1000, 128, 15
LEARNING_RATE = 2e-4
NUM_WORKERS = 0 if os.name == "nt" else 2
PIN_MEMORY = torch.cuda.is_available()
OUTPUT_ROOT = Path("outputs")

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")  # rezultatele se salvează imediat pe Google Drive, cu aceeași structură de directoare
    except Exception as exc:
        print("Google Drive nu a putut fi montat; se folosește stocarea temporară Colab:", exc)
    OUTPUT_ROOT = (Path("/content/drive/MyDrive/ddpm_lab4_outputs") if Path("/content/drive/MyDrive").exists()
                   else Path("/content/ddpm_lab4_outputs"))
EVAL_N = 1024  # imagini generate per configurație pentru FID/KID (mai mare = mai stabil, dar mai lent)
RESUME_FROM = {"linear": None, "cosine": None}  # Specificați calea unui checkpoint anterior pentru reluare
RUN_BONUS = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | dispozitiv: {DEVICE}")
if DEVICE.type == "cuda": print(torch.cuda.get_device_name(DEVICE))
else: print("ATENȚIE: CPU este potrivit pentru verificări rapide; antrenarea/sampler-ele pot fi foarte lente.")

## Persistența unei rulări

Fiecare execuție primește un director unic `outputs/run_<timestamp>_<id>/`, cu manifest, istoric de metrici actualizat atomic, log, checkpoint-uri complete, grile/ploturi și exporturi. Checkpoint-urile includ modelul, optimizerul, progresul, pierderile și stările RNG necesare reluării. Pentru reluare după resetarea kernelului, setați `RESUME_FROM['linear']` sau `RESUME_FROM['cosine']` la calea completă către checkpoint-ul dorit înainte de inițializarea rulării.

In [ ]:
@dataclass(frozen=True)
class RunPaths:
    root: Path
    checkpoints: Path
    samples: Path
    plots: Path
    exports: Path
    parameters: Path
    metrics: Path
    final_report: Path
    log: Path

def save_json_atomic(path, payload):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)

def create_run_paths(root=OUTPUT_ROOT):
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    run = root / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_{uuid.uuid4().hex[:6]}"
    run.mkdir()
    p = RunPaths(run, run/"checkpoints", run/"samples", run/"plots", run/"exports",
                 run/"parameters.json", run/"metrics.json", run/"final_report.json", run/"execution.log")
    for d in (p.checkpoints, p.samples, p.plots, p.exports): d.mkdir()
    return p

PATHS = create_run_paths()
logger = logging.getLogger(PATHS.root.name); logger.setLevel(logging.INFO); logger.handlers.clear()
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
fh = logging.FileHandler(PATHS.log, encoding="utf-8"); fh.setFormatter(fmt); logger.addHandler(fh)
sh = logging.StreamHandler(); sh.setFormatter(fmt); logger.addHandler(sh)
PARAMETERS = {
    "project": "Laboratorul 4 — DDPM pe Fashion-MNIST", "run_id": PATHS.root.name,
    "started_at": datetime.now(timezone.utc).isoformat(),
    "environment": {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
                    "numpy": np.__version__, "device": str(DEVICE),
                    "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None},
    "reproducibility": {"seed": SEED, "deterministic_mode": False},
    "configuration": {"dataset": "Fashion-MNIST", "image_shape": [1,32,32], "range": [-1,1],
                      "T": T, "batch_size": BATCH_SIZE, "epochs": EPOCHS, "optimizer": "Adam",
                      "learning_rate": LEARNING_RATE, "channels": [32,64,128],
                      "ddim_steps": [10,20,50,200,1000], "bonus_enabled": RUN_BONUS}}
save_json_atomic(PATHS.parameters, PARAMETERS)
METRICS = {"run_directory": str(PATHS.root), "metric_definitions": {
    "epoch_loss": "MSE medie per epocă; pierderile între programe nu sunt direct comparabile",
    "sampling_seconds": "timp măsurat cu sincronizare CUDA"}, "records": [], "status": "in_progress"}
save_json_atomic(PATHS.metrics, METRICS)
def add_metric(record):
    METRICS["records"].append(record); save_json_atomic(PATHS.metrics, METRICS)
logger.info("Rulare creată: %s", PATHS.root.resolve())
save_json_atomic(PATHS.final_report, {"run_id": PATHS.root.name, "status": "in_progress", "started_at": PARAMETERS["started_at"]})
def persist_uncaught_exception(exc_type, exc, tb):
    logger.error("Run failed", exc_info=(exc_type, exc, tb))
    METRICS["status"] = "failed"
    METRICS["failure"] = f"{exc_type.__name__}: {exc}"
    save_json_atomic(PATHS.metrics, METRICS)
    save_json_atomic(PATHS.final_report, {"run_id": PATHS.root.name, "status": "failed",
        "finished_at": datetime.now(timezone.utc).isoformat(), "failure": METRICS["failure"]})
sys.excepthook = persist_uncaught_exception
print("Output persistent:", PATHS.root.resolve())

## Sarcina 1 — Datele

Fashion-MNIST se completează cu `transforms.Pad(2)`, apoi se transformă și se normalizează în [-1,1]. DataLoader-ul are batch-uri de 128. Se verifică forma/intervalul și se afișează 16 imagini.

In [ ]:
transform = transforms.Compose([transforms.Pad(2), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
data_root = OUTPUT_ROOT / "data"; data_root.mkdir(parents=True, exist_ok=True)
train_dataset = datasets.FashionMNIST(str(data_root), train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(str(data_root), train=False, download=True, transform=transform)
def make_train_loader(seed=SEED):
    gen = torch.Generator().manual_seed(seed)
    return DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=gen,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS>0), gen
train_loader, _ = make_train_loader()
images, labels = next(iter(train_loader))
assert tuple(images.shape[1:]) == (1,32,32) and torch.isfinite(images).all()
assert images.min().item() >= -1.001 and images.max().item() <= 1.001
fig, axs = plt.subplots(4,4,figsize=(7,7))
for ax, im, label in zip(axs.flat, images[:16], labels[:16]):
    ax.imshow(im.squeeze().numpy()*0.5+0.5, cmap="gray", vmin=0, vmax=1)
    ax.set_title(train_dataset.classes[int(label)], fontsize=8); ax.axis("off")
fig.suptitle("Fashion-MNIST — batch de antrenare 32×32"); fig.tight_layout()
fig.savefig(PATHS.samples/"fashion_mnist_batch.png", dpi=140); plt.show()
print(f"Train: {len(train_dataset):,}; test: {len(test_dataset):,}; batch: {tuple(images.shape)}; interval: [{images.min():.3f}, {images.max():.3f}]")

## Sarcina 2 — Programele de zgomot și procesul direct

Se implementează β liniar 0,0001→0,02 și cosinus cu s=0,008; apoi αₜ=1−βₜ și ᾱₜ=cumprod(α). `q_sample` selectează direct orice t. Grila folosește aceeași imagine și aceeași ε pentru ambele programe. Pragul este primul t pentru care √ᾱₜ < 0,5.

In [ ]:
def make_schedule(name, T=T, device=DEVICE):
    if name == "linear":
        beta = torch.linspace(1e-4, 2e-2, T, dtype=torch.float32, device=device)
    elif name == "cosine":
        s = 0.008
        u = torch.arange(T+1, dtype=torch.float64, device=device)
        f = torch.cos(((u/T+s)/(1+s))*math.pi/2).square()
        abar = f/f[0]
        beta = (1-abar[1:]/abar[:-1]).clamp(max=0.999).float()
    else: raise ValueError("Numele programului trebuie să fie linear sau cosine")
    alpha = 1-beta; alpha_bar = torch.cumprod(alpha, dim=0)
    assert beta.shape == alpha.shape == alpha_bar.shape == (T,)
    assert torch.all((beta>0)&(beta<=0.999))
    return {"betas": beta, "alphas": alpha, "alpha_bars": alpha_bar}
SCHEDULES = {k: make_schedule(k) for k in ("linear","cosine")}

def extract(values, t, xshape):
    return values.gather(0,t.long()).reshape(t.shape[0], *([1]*(len(xshape)-1)))
def q_sample(x0, t, eps, schedule):
    ab = extract(schedule["alpha_bars"], t, x0.shape)
    return ab.sqrt()*x0 + (1-ab).sqrt()*eps

x0_demo = test_dataset[0][0].unsqueeze(0).to(DEVICE)
eps_demo = torch.randn(x0_demo.shape, device=DEVICE, generator=torch.Generator(device=DEVICE).manual_seed(SEED))
plot_steps = [0,50,100,250,500,750,999]
fig, axs = plt.subplots(2,7,figsize=(15,4)); threshold_steps = {}
for row,name in enumerate(("linear","cosine")):
    sch = SCHEDULES[name]
    for col,step in enumerate(plot_steps):
        xt = q_sample(x0_demo, torch.tensor([step],device=DEVICE), eps_demo, sch)[0,0].cpu()
        axs[row,col].imshow(xt,cmap="gray",vmin=-1,vmax=1); axs[row,col].set_title(f"t={step}"); axs[row,col].axis("off")
    axs[row,0].set_ylabel(name)
    below = torch.where(sch["alpha_bars"].sqrt()<0.5)[0]
    threshold_steps[name] = int(below[0]) if len(below) else None
fig.suptitle("q_sample — aceeași imagine și ε"); fig.tight_layout()
fig.savefig(PATHS.samples/"forward_process_grid.png",dpi=150); plt.show()
fig,ax=plt.subplots(figsize=(8,4))
for name,sch in SCHEDULES.items(): ax.plot(sch["alpha_bars"].sqrt().cpu().numpy(),label=name)
ax.axhline(.5,color="black",ls="--",label="prag 0,5")
ax.set(xlabel="t",ylabel="√ᾱₜ",title="Păstrarea semnalului"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
fig.savefig(PATHS.plots/"signal_retention.png",dpi=150); plt.show()
print("Primul t cu √ᾱₜ < 0,5:", threshold_steps)

## Sarcina 3 — U-Net mic condiționat de timp

Implementarea reproduce rețeaua din fișă: embedding sinusoidal, canale (32,64,128), blocuri reziduale, concatenări skip și zgomot prezis. În bloc, t este proiectat după prima convoluție; `h + self.skip(x)` este conexiunea reziduală. `s1/s2/s3` se concatenează în decoder.

In [ ]:
def time_embedding(t, dim=128):
    half = dim//2
    freqs = torch.exp(-math.log(10000)*torch.arange(half,device=t.device,dtype=torch.float32)/half)
    args = t.float()[:,None]*freqs[None,:]
    emb = torch.cat([torch.sin(args),torch.cos(args)],dim=1)
    return F.pad(emb,(0,1)) if dim%2 else emb

class ResBlock(nn.Module):
    def __init__(self,c_in,c_out,t_dim=128):
        super().__init__()
        self.norm1=nn.GroupNorm(8,c_in); self.conv1=nn.Conv2d(c_in,c_out,3,padding=1)
        self.t_proj=nn.Linear(t_dim,c_out); self.norm2=nn.GroupNorm(8,c_out)
        self.conv2=nn.Conv2d(c_out,c_out,3,padding=1)
        self.skip=nn.Conv2d(c_in,c_out,1) if c_in!=c_out else nn.Identity()
    def forward(self,x,temb):
        h=self.conv1(F.silu(self.norm1(x)))
        h=h+self.t_proj(temb)[:,:,None,None]  # condiționare cu pasul t
        h=self.conv2(F.silu(self.norm2(h)))
        return h+self.skip(x)  # conexiune reziduală

class UNet(nn.Module):
    def __init__(self,ch=(32,64,128),t_dim=128):
        super().__init__()
        self.t_mlp=nn.Sequential(nn.Linear(t_dim,t_dim),nn.SiLU(),nn.Linear(t_dim,t_dim))
        self.inp=nn.Conv2d(1,ch[0],3,padding=1)
        self.down1=ResBlock(ch[0],ch[0]); self.down2=ResBlock(ch[0],ch[1]); self.down3=ResBlock(ch[1],ch[2])
        self.mid=ResBlock(ch[2],ch[2])
        self.up3=ResBlock(2*ch[2],ch[1]); self.up2=ResBlock(2*ch[1],ch[0]); self.up1=ResBlock(2*ch[0],ch[0])
        self.out=nn.Conv2d(ch[0],1,1)
    def forward(self,x,t):
        temb=self.t_mlp(time_embedding(t)); x=self.inp(x)
        s1=self.down1(x,temb)  # skip la 32×32
        s2=self.down2(F.avg_pool2d(s1,2),temb)  # skip la 16×16
        s3=self.down3(F.avg_pool2d(s2,2),temb)  # skip la 8×8
        h=self.mid(F.avg_pool2d(s3,2),temb)  # 4×4
        up=lambda z:F.interpolate(z,scale_factor=2,mode="nearest")
        h=self.up3(torch.cat([up(h),s3],dim=1),temb)  # concatenare skip s3
        h=self.up2(torch.cat([up(h),s2],dim=1),temb)  # concatenare skip s2
        h=self.up1(torch.cat([up(h),s1],dim=1),temb)  # concatenare skip s1
        return self.out(h)  # ε prezis

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

check_model=UNet().to(DEVICE)
with torch.no_grad():
    xcheck=torch.randn(8,1,32,32,device=DEVICE); tcheck=torch.randint(0,T,(8,),device=DEVICE)
    ycheck=check_model(xcheck,tcheck)
assert ycheck.shape==xcheck.shape and torch.isfinite(ycheck).all()
parameter_count=count_parameters(check_model)
print("Formă intrare/ieșire:",tuple(xcheck.shape),tuple(ycheck.shape))
print(f"Parametri antrenabili: {parameter_count:,} ({parameter_count/1e6:.3f} milioane)")
del check_model,xcheck,tcheck,ycheck
if DEVICE.type=="cuda": torch.cuda.empty_cache()

## Sarcina 4 — Antrenare, checkpoint-uri și reluare

La fiecare batch se eșantionează t∈{0,…,999} și ε~N(0,I), se construiește xₜ cu `q_sample`, apoi se minimizează `F.mse_loss(εθ(xₜ,t), ε)`. Adam, lr=0,0002, batch=128, 15 epoci. După fiecare epocă se salvează checkpoint complet, snapshot și best checkpoint, iar metrics.json se actualizează atomic. Întreruperea în timpul unei epoci păstrează ultima epocă completă. La reluare într-un kernel nou, schimbați RESUME_FROM înainte de celula care creează rularea.

In [ ]:
def atomic_torch_save(payload,path):
    path=Path(path); tmp=path.with_suffix(path.suffix+".tmp"); torch.save(payload,tmp); tmp.replace(path)
def rng_state(loader_gen):
    return {"python":random.getstate(),"numpy":np.random.get_state(),"cpu":torch.get_rng_state(),
            "cuda":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "loader":loader_gen.get_state()}
def restore_rng(state,loader_gen):
    random.setstate(state["python"]); np.random.set_state(state["numpy"]); torch.set_rng_state(state["cpu"])
    if torch.cuda.is_available() and state["cuda"] is not None: torch.cuda.set_rng_state_all(state["cuda"])
    loader_gen.set_state(state["loader"])

def train_model(schedule_name,channels=(32,64,128),resume_from=None):
    schedule=SCHEDULES[schedule_name]; seed_everything(SEED)
    loader,loader_gen=make_train_loader(SEED); model=UNet(ch=channels).to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    losses=[]; start_epoch=0; best=float("inf")
    config={"schedule":schedule_name,"channels":list(channels),"batch_size":BATCH_SIZE,"learning_rate":LEARNING_RATE}
    if resume_from:
        cp=torch.load(resume_from,map_location=DEVICE,weights_only=False)
        if cp.get("configuration")!=config: raise ValueError(f"Checkpoint incompatibil: {cp.get('configuration')} != {config}")
        model.load_state_dict(cp["model_state"]); optimizer.load_state_dict(cp["optimizer_state"])
        losses=list(cp["losses"]); start_epoch=int(cp["epoch"]); best=float(cp["best_loss"])
        restore_rng(cp["rng_state"],loader_gen); logger.info("Reluare %s de la epoca %s",schedule_name,start_epoch+1)
    started=time.perf_counter()
    for epoch in range(start_epoch,EPOCHS):
        model.train(); total=0.; seen=0
        for x0,_ in loader:
            x0=x0.to(DEVICE,non_blocking=PIN_MEMORY); b=x0.shape[0]
            t=torch.randint(0,T,(b,),device=DEVICE); eps=torch.randn_like(x0)
            xt=q_sample(x0,t,eps,schedule); optimizer.zero_grad(set_to_none=True)
            loss=F.mse_loss(model(xt,t),eps); loss.backward(); optimizer.step()
            total+=loss.item()*b; seen+=b
        value=total/seen; losses.append(value); elapsed=time.perf_counter()-started
        add_metric({"step":epoch+1,"elapsed_seconds":elapsed,"metrics":{"epoch_loss":value,"schedule":schedule_name},"status":"completed"})
        logger.info("%s epoca %02d/%02d MSE=%.6f elapsed=%.1fs",schedule_name,epoch+1,EPOCHS,value,elapsed)
        state={"format_version":1,"schedule":schedule_name,"configuration":config,"epoch":epoch+1,
               "losses":losses,"best_loss":min(best,value),"model_state":model.state_dict(),
               "optimizer_state":optimizer.state_dict(),"rng_state":rng_state(loader_gen)}
        atomic_torch_save(state,PATHS.checkpoints/f"latest_{schedule_name}.pt")
        atomic_torch_save(state,PATHS.checkpoints/f"checkpoint_{schedule_name}_epoch_{epoch+1:02d}.pt")
        if value<best:
            best=value; state["best_loss"]=best
            atomic_torch_save(state,PATHS.checkpoints/f"best_{schedule_name}.pt")
    return model.eval(),losses,time.perf_counter()-started

models={}; loss_histories={}; training_seconds={}
for name in ("linear","cosine"):
    models[name],loss_histories[name],training_seconds[name]=train_model(name,resume_from=RESUME_FROM.get(name))
fig,ax=plt.subplots(figsize=(7,4))
for name,values in loss_histories.items(): ax.plot(range(1,len(values)+1),values,marker="o",label=name)
ax.set(xlabel="Epocă",ylabel="MSE medie",title="Pierdere pe epoci"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
fig.savefig(PATHS.plots/"training_losses.png",dpi=150); plt.show()
print("Durate măsurate (s):",training_seconds)
print("Pierderile celor două programe nu sunt direct comparabile: distribuțiile nivelurilor de zgomot diferă.")

> **Checkpoint-uri:** găsiți `latest_linear.pt` și `latest_cosine.pt` în directorul rulării. Pentru reluare, folosiți căile respective într-o rulare nouă. Epoca și configurația sunt verificate înainte de încărcare.

## Sarcina 5 — Eșantionare DDPM

Sampler-ul implementează algoritmul 2, cu σ²ₜ=βₜ, t=999→0 și fără zgomot la t=0. Se generează 64 imagini și se salvează opt stări pentru prima imagine.

In [ ]:
@torch.inference_mode()
def sample_ddpm(model,schedule,shape,seed=SEED,capture=False):
    model.eval(); gen=torch.Generator(device=DEVICE).manual_seed(seed)
    x=torch.randn(shape,device=DEVICE,generator=gen)
    states=[(T,x[0:1].cpu())] if capture else None
    capture_at={850,700,550,400,250,100}
    for step in range(T-1,-1,-1):
        t=torch.full((shape[0],),step,dtype=torch.long,device=DEVICE)
        beta=schedule["betas"][step]; alpha=schedule["alphas"][step]; ab=schedule["alpha_bars"][step]
        eps=model(x,t); mean=(x-beta/torch.sqrt(1-ab)*eps)/torch.sqrt(alpha)
        x=mean+torch.sqrt(beta)*torch.randn(x.shape,device=DEVICE,generator=gen) if step>0 else mean
        if capture and step in capture_at: states.append((step,x[0:1].cpu()))
    if capture: states.append((0,x[0:1].cpu()))
    return x.clamp(-1,1),states

def show_grid(images,title,save_path=None,ncol=8):
    images=images.detach().cpu().clamp(-1,1); n=len(images); nrow=math.ceil(n/ncol)
    fig,axs=plt.subplots(nrow,ncol,figsize=(ncol*1.45,nrow*1.45),squeeze=False)
    for i,ax in enumerate(axs.flat):
        ax.axis("off")
        if i<n: ax.imshow(images[i,0],cmap="gray",vmin=-1,vmax=1)
    fig.suptitle(title); fig.tight_layout()
    if save_path: fig.savefig(save_path,dpi=150)
    plt.show(); return fig

if "linear" not in models: raise RuntimeError("Rulați antrenarea înainte de eșantionare.")
if DEVICE.type=="cuda": torch.cuda.synchronize()
start=time.perf_counter()
ddpm_images,ddpm_states=sample_ddpm(models["linear"],SCHEDULES["linear"],(64,1,32,32),seed=SEED+1,capture=True)
if DEVICE.type=="cuda": torch.cuda.synchronize()
ddpm_seconds=time.perf_counter()-start
show_grid(ddpm_images,"DDPM — program liniar, 64 imagini",PATHS.samples/"ddpm_linear_1000.png")
fig,axs=plt.subplots(1,len(ddpm_states),figsize=(16,2.5))
for ax,(label,state) in zip(axs,ddpm_states):
    ax.imshow(state[0,0],cmap="gray",vmin=-1,vmax=1); ax.set_title(f"t≈{label}"); ax.axis("off")
fig.suptitle("Stări intermediare DDPM"); fig.tight_layout()
fig.savefig(PATHS.samples/"ddpm_trajectory.png",dpi=150); plt.show()
print(f"DDPM 64 imagini: {ddpm_seconds:.2f} s")

## Sarcina 6 — DDIM și cronometrare

DDIM determinist la 10/20/50/200/1000 pași. Toate variantele folosesc aceeași latentă inițială; cronometrul sincronizează CUDA înainte de început și înaintea citirii finale. Calitatea se măsoară automat (celula următoare): FID și KID față de setul de test Fashion-MNIST, iar nota 1–5 se derivă din FID între un plafon (real vs. real) și o limită inferioară (zgomot). Inspectarea vizuală a grilelor rămâne recomandată, dar nu este obligatorie pentru raport.

In [ ]:
@torch.inference_mode()
def sample_ddim(model,schedule,initial_noise,n_steps):
    if not 1<=n_steps<=T: raise ValueError(f"n_steps trebuie în [1,{T}]")
    model.eval(); x=initial_noise.clone().to(DEVICE)
    ts=torch.unique_consecutive(torch.linspace(T-1,0,n_steps,device=DEVICE).round().long())
    if len(ts)!=n_steps: raise RuntimeError(f"Pași unici: {len(ts)}, ceruți: {n_steps}")
    abar=schedule["alpha_bars"]
    for i,st in enumerate(ts):
        step=int(st); t=torch.full((x.shape[0],),step,dtype=torch.long,device=DEVICE)
        eps=model(x,t); ab=abar[step]; x0=(x-torch.sqrt(1-ab)*eps)/torch.sqrt(ab)
        if i==len(ts)-1: x=x0
        else:
            ab_next=abar[int(ts[i+1])]
            x=torch.sqrt(ab_next)*x0+torch.sqrt(1-ab_next)*eps
    return x.clamp(-1,1)
def synchronize():
    if DEVICE.type=="cuda": torch.cuda.synchronize(DEVICE)

gen=torch.Generator(device=DEVICE).manual_seed(SEED+1)
shared_latents=torch.randn((64,1,32,32),device=DEVICE,generator=gen)
sampling_results={"ddpm_1000":ddpm_images}; sampling_seconds={"ddpm_1000":ddpm_seconds}
for n in (10,20,50,200,1000):
    synchronize(); start=time.perf_counter()
    result=sample_ddim(models["linear"],SCHEDULES["linear"],shared_latents,n)
    synchronize(); elapsed=time.perf_counter()-start
    key=f"ddim_{n}"; sampling_results[key]=result; sampling_seconds[key]=elapsed
    show_grid(result,f"DDIM liniar — {n} pași",PATHS.samples/f"ddim_linear_{n:04d}.png")
    add_metric({"step":f"sampling_{key}","elapsed_seconds":elapsed,
                "metrics":{"sampling_seconds":elapsed,"method":"DDIM","steps":n,"images":64,"schedule":"linear"},"status":"completed"})
    logger.info("DDIM liniar %d pași: %.3f s",n,elapsed)

# Notele și observațiile sunt completate automat din FID/KID în celula următoare.
quality_ratings={"ddpm_1000":None,"ddim_1000":None,"ddim_200":None,"ddim_50":None,"ddim_20":None,"ddim_10":None}
visual_observations={key:"" for key in quality_ratings}


### Evaluare automată: FID și KID

În locul notelor vizuale manuale, fiecare configurație generează `EVAL_N` imagini care sunt comparate cu setul de test. FID/KID mai mici înseamnă distribuție mai apropiată de date. Atenție: Inception este antrenat pe ImageNet, iar FID la N mic este părtinitor; valorile se compară doar relativ, în aceeași rulare. Cronometrarea de mai sus este separată de evaluare.

In [ ]:
%pip install -q torchmetrics torch-fidelity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

def _to_rgb_unit(x):
    return ((x.detach().clamp(-1, 1) + 1) / 2).repeat(1, 3, 1, 1).float()

fid_metric = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(DEVICE)
kid_metric = KernelInceptionDistance(subset_size=min(100, EVAL_N // 2), normalize=True, reset_real_features=False).to(DEVICE)

def _push(x, real):
    for i in range(0, len(x), 128):
        batch = _to_rgb_unit(x[i:i+128]).to(DEVICE)
        fid_metric.update(batch, real=real); kid_metric.update(batch, real=real)

def _score_against_reference(images):
    fid_metric.reset(); kid_metric.reset()  # reset_real_features=False păstrează statisticile reale
    _push(images, real=False)
    kid_mean, kid_std = kid_metric.compute()
    return float(fid_metric.compute()), float(kid_mean), float(kid_std)

# Referința reală: imagini de test din afara subsetului folosit pentru plafon.
REF_N = 5000
ref_images = torch.stack([test_dataset[i][0] for i in range(EVAL_N, EVAL_N + REF_N)])
_push(ref_images, real=True)
floor_images = torch.stack([test_dataset[i][0] for i in range(EVAL_N)])
FID_FLOOR, KID_FLOOR, _ = _score_against_reference(floor_images)
noise_gen = torch.Generator().manual_seed(SEED + 7)
FID_NOISE, KID_NOISE, _ = _score_against_reference(torch.randn((EVAL_N, 1, 32, 32), generator=noise_gen).clamp(-1, 1))
print(f"Plafon FID (real vs. real, N={EVAL_N}): {FID_FLOOR:.2f} | limită inferioară (zgomot): {FID_NOISE:.2f}")

def auto_score(fid):
    ratio = (fid - FID_FLOOR) / max(FID_NOISE - FID_FLOOR, 1e-9)
    return round(float(5 - 4 * min(max(ratio, 0.0), 1.0)), 2)

def generate_eval(model, schedule, method, n_steps):
    out = []
    for i in range(0, EVAL_N, 256):
        b = min(256, EVAL_N - i)
        if method == "ddpm":
            x, _ = sample_ddpm(model, schedule, (b, 1, 32, 32), seed=SEED + 1000 + i)
        else:
            g = torch.Generator(device=DEVICE).manual_seed(SEED + 1000 + i)  # același zgomot pentru toți pașii DDIM
            x = sample_ddim(model, schedule, torch.randn((b, 1, 32, 32), device=DEVICE, generator=g), n_steps)
        out.append(x.cpu())
    return torch.cat(out)

def evaluate_config(label, model, schedule, method, n_steps):
    images = generate_eval(model, schedule, method, n_steps)
    fid, kid, kid_std = _score_against_reference(images)
    result = {"fid": fid, "kid_mean": kid, "kid_std": kid_std, "auto_score": auto_score(fid), "n_images": EVAL_N}
    add_metric({"step": f"eval_{label}", "metrics": result, "status": "completed"})
    logger.info("%s: FID=%.2f KID=%.4f nota=%.2f", label, fid, kid, result["auto_score"])
    return result

fid_results = {}
for key in ("ddpm_1000", "ddim_1000", "ddim_200", "ddim_50", "ddim_20", "ddim_10"):
    method, steps = key.rsplit("_", 1)
    fid_results[key] = evaluate_config(key, models["linear"], SCHEDULES["linear"], method, int(steps))
    quality_ratings[key] = fid_results[key]["auto_score"]
    visual_observations[key] = f"auto: FID={fid_results[key]['fid']:.1f}, KID={fid_results[key]['kid_mean']:.4f}"
METRICS["fid_reference"] = {"floor_fid": FID_FLOOR, "noise_fid": FID_NOISE, "eval_n": EVAL_N, "reference_n": REF_N}
save_json_atomic(PATHS.metrics, METRICS)


In [ ]:
import pandas as pd
rows=[]
for key in ("ddpm_1000","ddim_1000","ddim_200","ddim_50","ddim_20","ddim_10"):
    method,steps=key.rsplit("_",1)
    rows.append({"Metodă":method.upper(),"Pași":int(steps),"Timp pentru 64 imagini (s)":sampling_seconds.get(key),
                 "FID ↓":round(fid_results[key]["fid"],2),"KID ↓":round(fid_results[key]["kid_mean"],4),
                 "Calitate (1–5, din FID)":quality_ratings[key],"Observații":visual_observations[key]})
sampling_table=pd.DataFrame(rows); display(sampling_table)
sampling_table.to_csv(PATHS.exports/"sampling_comparison.csv",index=False,encoding="utf-8-sig")


## Sarcina 7 — Liniar versus cosinus

Se compară cele două modele cu DDIM la 50 și 20 pași, folosind aceeași latentă de pornire. Curbele de pierdere se trasează împreună; diferențele de loss nu sunt o comparație directă a calității. Completați notele de calitate după inspecția imaginilor.

In [ ]:
comparison_samples={}
for name in ("linear","cosine"):
    if name not in models: raise RuntimeError(f"Lipsește modelul {name}; rulați antrenarea.")
    for n in (50,20):
        result=sample_ddim(models[name],SCHEDULES[name],shared_latents,n)
        comparison_samples[(name,n)]=result.detach().cpu()
        show_grid(result,f"DDIM {name} — {n} pași, latentă comună",PATHS.samples/f"comparison_{name}_{n:02d}.png")
fig,axs=plt.subplots(1,2,figsize=(12,4))
for name,values in loss_histories.items(): axs[0].plot(range(1,len(values)+1),values,marker="o",label=name)
axs[0].set(xlabel="Epocă",ylabel="MSE",title="Pierderi"); axs[0].grid(alpha=.25); axs[0].legend()
for name,sch in SCHEDULES.items(): axs[1].plot(sch["alpha_bars"].sqrt().cpu().numpy(),label=name)
axs[1].set(xlabel="t",ylabel="√ᾱₜ",title="Păstrarea semnalului"); axs[1].grid(alpha=.25); axs[1].legend()
fig.tight_layout(); fig.savefig(PATHS.plots/"linear_vs_cosine.png",dpi=150); plt.show()
comparison_quality={}; comparison_notes={}; comparison_fid={}
for name in ("linear","cosine"):
    for n in (50,20):
        res=evaluate_config(f"{name}_{n}",models[name],SCHEDULES[name],"ddim",n)
        comparison_fid[f"{name}_{n}"]=res; comparison_quality[f"{name}_{n}"]=res["auto_score"]
        comparison_notes[f"{name}_{n}"]=f"auto: FID={res['fid']:.1f}, KID={res['kid_mean']:.4f}"
comparison_table=pd.DataFrame([{"Program":name,"Pași DDIM":n,"FID ↓":round(comparison_fid[f"{name}_{n}"]["fid"],2),
                                "KID ↓":round(comparison_fid[f"{name}_{n}"]["kid_mean"],4),
                                "Calitate (1–5, din FID)":comparison_quality[f"{name}_{n}"]}
                               for n in (50,20) for name in ("linear","cosine")])
display(comparison_table)
comparison_table.to_csv(PATHS.exports/"schedule_comparison.csv",index=False,encoding="utf-8-sig")


## Întrebări de control — completați răspunsurile proprii

Completați valorile de mai jos. La export, câmpurile rămase goale apar ca necompletate.

In [ ]:
control_questions=[
"1. De ce procesul direct nu are parametri învățați? Ce rol are în antrenare?",
"2. De ce putem sări direct la pasul t, fără să parcurgem pașii anteriori?",
"3. De ce rețeaua primește și numărul pasului t? Ce s-ar întâmpla fără el?",
"4. De ce generarea e mult mai lentă decât un pas de antrenare?",
"5. Ce diferență vedeți între programul liniar și cel cosinus la Sarcinile 2 și 7? Cum o explicați?",
"6. Ce câștigați și ce pierdeți când reduceți pașii DDIM de la 1000 la 10?",
"7. De ce pierderile celor două modele de la Sarcina 7 nu se pot compara direct?"]
control_answers={
control_questions[0]:"Pentru că procesul direct este doar adăugare de zgomot, nu are parametri de optimizat. Rolul lui e să ofere date de antrenare pentru procesul invers.",
control_questions[1]:"Pentru că zgomotul se poate calcula direct la pasul t prin formula analitică, nu e nevoie să simulăm fiecare pas.",
control_questions[2]:"Numărul pasului t indică intensitatea zgomotului. Fără el, rețeaua nu ar ști cât zgomot să elimine.",
control_questions[3]:"Generarea e iterativă, fiecare pas depinde de rezultatul anterior. În antrenare, toate pașii se pot calcula în paralel.",
control_questions[4]:"Programul cosinus produce o distribuție mai uniformă a zgomotului, liniarul e mai abrupt. Diferența se vede în calitatea reconstrucției.",
control_questions[5]:"Reducerea pașilor accelerează generarea, dar scade fidelitatea și stabilitatea imaginilor. Se câștigă viteză, se pierde calitate.",
control_questions[6]:"Pentru că sunt antrenate cu zgomote diferite și obiective diferite, pierderile nu sunt direct comparabile."
}
for q in control_questions:
    print(q,"\nRăspuns:",control_answers[q],"\n")


## Bonus opțional (+1) — U-Net mai mic

Bonusul este oprit implicit (`RUN_BONUS=False`). Pentru efectuare, setați `RUN_BONUS=True` în configurație și rulați notebook-ul de la început, apoi celula de bonus. Antrenați `UNet(ch=(16,32,64))) cu seed și setări identice; comparați numărul de parametri, timpul efectiv și calitatea observată.

In [ ]:
RUN_BONUS=True
if RUN_BONUS:
    bonus_start=time.perf_counter()
    bonus_model,bonus_losses,bonus_training_seconds=train_model("linear",channels=(16,32,64),resume_from=None)
    bonus_params=count_parameters(bonus_model)
    print(f"Bonus: {bonus_params:,} parametri; timp măsurat {bonus_training_seconds:.1f} s")
    bonus_images=sample_ddim(bonus_model,SCHEDULES["linear"],shared_latents,50)
    show_grid(bonus_images,"Bonus — U-Net (16,32,64), DDIM 50",PATHS.samples/"bonus_unet_16_32_64.png")
else: print("Bonus dezactivat.")

## Export raport PDF (4 pagini, doar din rezultate reale)

Rula?i aceast? celul? **numai dup?** antrenarea ambelor modele (15 epoci fiecare), DDPM/DDIM, compara?ia liniar/cosinus ?i completarea manual? a notelor 1?5, observa?iilor ?i r?spunsurilor din celulele anterioare.

`export_report_pdf()` verific? ?nainte de a scrie PDF-ul: pierderi pe 15 epoci ?i checkpoint-uri finale, timpi pozitivi pentru toate cele 6 metode, 8 st?ri DDPM, cele 4 grile comparative, figurile salvate ?n rulare, note ?i r?spunsuri completate. Dac? lipse?te ceva, apare o eroare clar? ?i **nu se creeaz? niciun PDF**. Pentru o ciorn? explicit marcat?, folosi?i `export_report_pdf(allow_incomplete=True)`; aceasta tolereaz? doar note/observa?ii/r?spunsuri lips? (nu rezultate experimentale) ?i produce `*_DRAFT.pdf`, f?r? a marca rularea ca finalizat?.

PDF-ul ?i un `report_manifest.json` sunt salvate ?n `exports/` al rul?rii curente (`PATHS.exports`) ?i nu suprascriu fi?iere existente.

In [ ]:
import textwrap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.image import imread

class ReportInputError(RuntimeError):
    """Rezultatele necesare raportului lipsesc sau sunt invalide."""

def _blank(value):
    return value is None or (isinstance(value, str) and not value.strip())

def _fmt(value):
    return "necompletat" if _blank(value) else str(value)

def _finite_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)

def _collect_report_inputs(allow_incomplete=False):
    """?ntoarce (erori critice, avertismente despre c?mpuri manuale necompletate)."""
    errors, manual = [], []
    required = ("models", "loss_histories", "training_seconds", "threshold_steps", "ddpm_states",
                "sampling_seconds", "sampling_results", "comparison_samples", "quality_ratings",
                "visual_observations", "comparison_quality", "comparison_notes",
                "control_questions", "control_answers", "fid_results", "comparison_fid")
    missing = [n for n in required if n not in globals()]
    if missing:
        return [f"Variabile lips? (rula?i celulele anterioare): {', '.join(missing)}"], manual

    for name in ("linear", "cosine"):
        if name not in models: errors.append(f"Modelul '{name}' nu este antrenat.")
        history = loss_histories.get(name)
        if not history or len(history) != EPOCHS or not all(_finite_number(v) for v in history):
            errors.append(f"Istoricul de pierdere '{name}' trebuie s? aib? {EPOCHS} valori finite; are {None if not history else len(history)}.")
        if not _finite_number(training_seconds.get(name)) or training_seconds[name] <= 0:
            errors.append(f"Timpul de antrenare m?surat pentru '{name}' lipse?te.")
        if threshold_steps.get(name) is None:
            errors.append(f"Pragul ???<0,5 pentru '{name}' nu a fost calculat.")
        for ckpt in (f"latest_{name}.pt", f"checkpoint_{name}_epoch_{EPOCHS:02d}.pt"):
            if not (PATHS.checkpoints / ckpt).is_file():
                errors.append(f"Checkpoint lips? ?n rularea curent?: checkpoints/{ckpt}")

    sampling_keys = ("ddpm_1000", "ddim_1000", "ddim_200", "ddim_50", "ddim_20", "ddim_10")
    for key in sampling_keys:
        if not _finite_number(sampling_seconds.get(key)) or sampling_seconds[key] <= 0:
            errors.append(f"Timpul de e?antionare '{key}' lipse?te sau este invalid.")
        images = sampling_results.get(key)
        if images is None or tuple(images.shape) != (64, 1, 32, 32) or not torch.isfinite(images).all():
            errors.append(f"Imaginile '{key}' lipsesc sau nu au forma (64, 1, 32, 32).")
    if len(ddpm_states) != 8:
        errors.append(f"?irul DDPM trebuie s? aib? 8 st?ri; are {len(ddpm_states)}.")
    for name in ("linear", "cosine"):
        for steps in (50, 20):
            images = comparison_samples.get((name, steps))
            if images is None or tuple(images.shape) != (64, 1, 32, 32) or not torch.isfinite(images).all():
                errors.append(f"Grila comparativ? ({name}, DDIM {steps}) lipse?te sau este invalid?.")

    required_files = [PATHS.samples / f for f in ("forward_process_grid.png", "ddpm_linear_1000.png", "ddpm_trajectory.png")]
    required_files += [PATHS.plots / f for f in ("signal_retention.png", "training_losses.png", "linear_vs_cosine.png")]
    for path in required_files:
        if not path.is_file(): errors.append(f"Figur? lips?: {path.relative_to(PATHS.root)}")

    for key in sampling_keys:
        if key not in fid_results or not _finite_number(fid_results[key].get("fid")):
            errors.append(f"FID lipsește pentru {key} (rulați celula de evaluare automată).")
    for key in ("linear_50", "cosine_50", "linear_20", "cosine_20"):
        if key not in comparison_fid or not _finite_number(comparison_fid[key].get("fid")):
            errors.append(f"FID lipsește pentru comparația {key}.")
    for question in control_questions:
        if _blank(control_answers.get(question)): manual.append(f"r?spuns: {question[:40]}?")
    return errors, manual

def _tile(images):
    batch = images.detach().cpu().clamp(-1, 1)[:, 0].numpy()
    return batch.reshape(8, 8, 32, 32).transpose(0, 2, 1, 3).reshape(256, 256)

def _text_page(fig, lines, top=0.96, size=8.5, step=0.0158, left=0.07, color="black"):
    y = top
    for line in lines:
        fig.text(left, y, line, fontsize=size, va="top", family="monospace", color=color)
        y -= step
    return y

def export_report_pdf(allow_incomplete=False):
    errors, manual = _collect_report_inputs(allow_incomplete)
    if errors or (manual and not allow_incomplete):
        parts = []
        if errors: parts.append("Rezultate experimentale lips?/invalide:\n - " + "\n - ".join(errors))
        if manual and not allow_incomplete:
            parts.append("C?mpuri manuale necompletate:\n - " + "\n - ".join(manual))
        raise ReportInputError("Raportul PDF NU a fost creat.\n" + "\n".join(parts))

    draft = bool(manual)
    stem = "raport_laborator_4_ddpm" + ("_DRAFT" if draft else "")
    pdf_path = PATHS.exports / f"{stem}.pdf"
    counter = 2
    while pdf_path.exists():
        pdf_path = PATHS.exports / f"{stem}_{counter}.pdf"; counter += 1
    tmp_path = pdf_path.with_suffix(".pdf.tmp")
    banner = "CIORN? ? c?mpuri manuale necompletate" if draft else None
    page_size = (8.27, 11.69)
    tail = f"Run {PATHS.root.name}"

    def finish(fig, pdf, number):
        fig.text(0.07, 0.015, f"{tail} | pagina {number}", fontsize=7, color="#555555")
        if banner: fig.text(0.93, 0.015, banner, fontsize=7, color="#b00020", ha="right")
        pdf.savefig(fig); plt.close(fig)

    try:
        with PdfPages(tmp_path) as pdf:
            # Pagina 1: configura?ie, procesul direct, semnal re?inut.
            fig = plt.figure(figsize=page_size)
            fig.text(0.07, 0.97, "Laboratorul 4 ? DDPM de la zero (Fashion-MNIST)", fontsize=16, weight="bold", va="top")
            n_params = count_parameters(models["linear"])
            info = [
                f"Dispozitiv: {DEVICE} | GPU: {PARAMETERS['environment']['gpu']} | PyTorch {torch.__version__} | seed {SEED}",
                f"Date: Fashion-MNIST 32x32, [-1,1], batch {BATCH_SIZE} | T={T} | Adam lr={LEARNING_RATE} | epoci/model={EPOCHS}",
                f"U-Net (32,64,128): {n_params:,} parametri antrenabili",
                f"Primul t cu sqrt(abar_t) < 0.5: linear={threshold_steps['linear']}, cosine={threshold_steps['cosine']}",
                f"Timp antrenare m?surat (s): linear={training_seconds['linear']:.1f}, cosine={training_seconds['cosine']:.1f}",
                f"Pierdere final? (epoca {EPOCHS}): linear={loss_histories['linear'][-1]:.5f}, cosine={loss_histories['cosine'][-1]:.5f}",
            ]
            _text_page(fig, info, top=0.935, size=8.5)
            ax = fig.add_axes([0.07, 0.62, 0.86, 0.21]); ax.imshow(imread(PATHS.samples / "forward_process_grid.png")); ax.axis("off")
            ax.set_title("Proces direct: r?ndul 1 liniar, r?ndul 2 cosinus (aceea?i imagine ?i ?)", fontsize=9)
            ax = fig.add_axes([0.07, 0.30, 0.86, 0.29]); ax.imshow(imread(PATHS.plots / "signal_retention.png")); ax.axis("off")
            note = textwrap.wrap("Pierderile celor dou? programe nu sunt direct comparabile: mediile se fac peste distribu?ii diferite ale nivelurilor de zgomot; compararea se face pe imaginile generate.", 110)
            _text_page(fig, note, top=0.27, size=8.5)
            finish(fig, pdf, 1)

            # Pagina 2: pierderi ?i DDPM.
            fig = plt.figure(figsize=page_size)
            fig.text(0.07, 0.97, "Antrenare ?i generare DDPM (program liniar)", fontsize=14, weight="bold", va="top")
            ax = fig.add_axes([0.07, 0.69, 0.86, 0.25]); ax.imshow(imread(PATHS.plots / "training_losses.png")); ax.axis("off")
            ax = fig.add_axes([0.20, 0.30, 0.60, 0.36]); ax.imshow(_tile(sampling_results["ddpm_1000"]), cmap="gray", vmin=-1, vmax=1); ax.axis("off")
            ax.set_title(f"DDPM 1000 pa?i, 64 imagini ({sampling_seconds['ddpm_1000']:.2f} s)", fontsize=9)
            ax = fig.add_axes([0.07, 0.12, 0.86, 0.14]); ax.imshow(imread(PATHS.samples / "ddpm_trajectory.png")); ax.axis("off")
            ax.set_title("St?ri intermediare (de la zgomot la rezultat)", fontsize=9)
            finish(fig, pdf, 2)

            # Pagina 3: tabel DDPM/DDIM ?i compara?ia liniar/cosinus.
            fig = plt.figure(figsize=page_size)
            fig.text(0.07, 0.97, "Timpi DDPM/DDIM ?i compara?ia liniar vs. cosinus", fontsize=14, weight="bold", va="top")
            lines = [f"{'Metodă':<8}{'Pași':>6}{'Timp 64 (s)':>12}{'FID':>9}{'KID':>9}{'Nota 1-5':>10}"]
            for key in ("ddpm_1000", "ddim_1000", "ddim_200", "ddim_50", "ddim_20", "ddim_10"):
                method, steps = key.rsplit("_", 1); r = fid_results[key]
                lines.append(f"{method.upper():<8}{steps:>6}{sampling_seconds[key]:>12.3f}{r['fid']:>9.1f}{r['kid_mean']:>9.4f}{r['auto_score']:>10.2f}")
            lines.append(f"Nota = 5 - 4*(FID-{FID_FLOOR:.1f})/({FID_NOISE:.1f}-{FID_FLOOR:.1f}), limitată la [1,5]; N={EVAL_N} imagini/config.")
            end = _text_page(fig, lines, top=0.935, size=7.5)
            positions = {("linear", 50): (0.07, 0.50), ("cosine", 50): (0.52, 0.50), ("linear", 20): (0.07, 0.19), ("cosine", 20): (0.52, 0.19)}
            for (name, steps), (left, bottom) in positions.items():
                ax = fig.add_axes([left, bottom, 0.41, 0.27]); ax.imshow(_tile(comparison_samples[(name, steps)]), cmap="gray", vmin=-1, vmax=1); ax.axis("off")
                ax.set_title(f"{name}, DDIM {steps} ? nota {_fmt(comparison_quality.get(f'{name}_{steps}'))}", fontsize=9)
            comp = [f"{k}: {textwrap.shorten(_fmt(v), 40, placeholder='?')}" for k, v in comparison_notes.items()]
            _text_page(fig, [" | ".join(comp[:2]), " | ".join(comp[2:])], top=0.14, size=7)
            finish(fig, pdf, 3)

            # Pagina 4: r?spunsuri la ?ntreb?rile de control.
            fig = plt.figure(figsize=page_size)
            fig.text(0.07, 0.97, "R?spunsuri la ?ntreb?rile de control", fontsize=14, weight="bold", va="top")
            body = []
            for question in control_questions:
                body += textwrap.wrap(question, 108) + [""]
                body += ["  " + w for w in textwrap.wrap(_fmt(control_answers.get(question)), 104)] + [""]
            capacity = 80
            if len(body) > capacity:
                raise ReportInputError(f"R?spunsurile ocup? {len(body)} linii; limita unei pagini este {capacity}. Scurta?i-le pentru a p?stra raportul ?n 4 pagini.")
            _text_page(fig, body, top=0.94, size=8, step=0.0113)
            finish(fig, pdf, 4)
            page_count = pdf.get_pagecount()
        assert 2 <= page_count <= 4, f"Num?r de pagini invalid: {page_count}"
        tmp_path.replace(pdf_path)
    except BaseException:
        if tmp_path.exists(): tmp_path.unlink()
        raise

    manifest = {"pdf": str(pdf_path.relative_to(PATHS.root)), "pages": page_count, "draft": draft,
                "missing_manual_fields": manual, "created_at": datetime.now(timezone.utc).isoformat(),
                "sources": {"training_seconds": training_seconds, "threshold_steps": threshold_steps,
                            "sampling_seconds": sampling_seconds, "quality_ratings": quality_ratings,
                            "comparison_quality": comparison_quality, "fid_results": fid_results, "comparison_fid": comparison_fid}}
    save_json_atomic(PATHS.exports / (pdf_path.stem + "_manifest.json"), manifest)
    if not draft:
        METRICS["status"] = "completed"; METRICS["completed_at"] = manifest["created_at"]
        save_json_atomic(PATHS.metrics, METRICS)
        save_json_atomic(PATHS.final_report, {
            "run_id": PATHS.root.name, "status": "completed", "finished_at": manifest["created_at"],
            "selected_artifact": manifest["pdf"],
            "selection": {"criterion": "modelele finale (epoca 15) ale acestei rul?ri; checkpoint-urile latest sunt p?strate"},
            "final_metrics": {"training_seconds": training_seconds, "final_losses": {k: v[-1] for k, v in loss_histories.items()},
                              "threshold_steps": threshold_steps, "sampling_seconds": sampling_seconds,
                              "quality_ratings": quality_ratings, "comparison_quality": comparison_quality},
            "report_pdf": manifest["pdf"]})
    logger.info("Raport PDF (%d pagini%s): %s", page_count, ", CIORN?" if draft else "", pdf_path)
    print(f"Raport PDF: {pdf_path.resolve()} ({page_count} pagini)")
    return pdf_path

report_path = export_report_pdf()  # e?ueaz? clar dac? lipsesc rezultate; allow_incomplete=True doar pentru ciorn?

## Salvare locală

Artefactele se scriu imediat în `PATHS.root` (pe Drive, dacă este montat). Celula următoare creează o arhivă cu structura `run_.../...` și o descarcă pe PC; implicit fără checkpoint-uri (mari). Setați `INCLUDE_CHECKPOINTS=True` pentru arhiva completă.

In [ ]:

import shutil, zipfile

def export_run_zip(root, include_checkpoints=False, max_checkpoint_mb=None, download=True):
    """Arhivează directorul rulării păstrând structura și, în Colab, îl descarcă pe PC-ul local."""
    root = Path(root).resolve(); base = root.parent
    out = base / (root.name + ("_complet" if include_checkpoints else "") + ".zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(root.rglob("*")):
            if not f.is_file() or f.suffix == ".tmp": continue
            if not include_checkpoints and f.suffix in (".pt", ".pth", ".ckpt", ".safetensors"): continue
            zf.write(f, f.relative_to(base).as_posix())
    print(f"Arhivă: {out} ({out.stat().st_size/1e6:.1f} MB)")
    if download and Path("/content").exists():
        from google.colab import files
        files.download(str(out))
    return out

INCLUDE_CHECKPOINTS = False
export_run_zip(PATHS.root, include_checkpoints=INCLUDE_CHECKPOINTS)


## Checklist pentru predare

- [ ] Grila forward cu două programe, graficul √ᾱₜ și pragurile măsurate.
- [ ] Codul U-Net comentat, forma și numărul real de parametri.
- [ ] Grafice loss, checkpoint-uri și timpi efectiv măsurați.
- [ ] Grilă DDPM 64 imagini și opt stări intermediare.
- [ ] Tabel DDPM/DDIM cu timpi, FID/KID și nota derivată automat.
- [ ] Comparație liniar/cosinus la DDIM 50/20, latentă comună și grile.
- [ ] Răspunsuri proprii la cele 7 întrebări.
- [ ] Raport PDF de 2–4 pagini, produs din rezultatele acestei rulări.
- [ ] Bonus numai dacă a fost efectiv rulat.

Directorul complet este indicat de `PATHS.root` (pe Google Drive dacă a fost montat; arhiva de mai jos îl descarcă local cu aceeași structură). Includeți raportul și artefactele necesare din `exports/`, `samples/`, `plots/` și checkpoint-urile.